# 🎭 Unified Moshi + AvatarForcing Pipeline
### RunPod Notebook — Talking-Head Video Generation

**Workflow:**
1. User audio → **Moshi** (Helium LM) → 8 response audio tokens per step + raw PCM
2. Response tokens → **moshi-wav2vec-bridge** → 10752-dim wav2vec-like embeddings
3. Embeddings + reference image → **AvatarForcing** diffusion → silent talking-head video
4. Silent video + Moshi raw audio → **ffmpeg** → final `.mp4`

**Latency gain:** Bridge converts discrete tokens *before* Mimi decodes them to waveform.

```
User Audio → [Moshi] → 8 tokens/step
                  │               └─► [Mimi Decoder] → raw PCM
                  └─► [Bridge]  → audio_emb (T+1, 10752)
                                      │
                           [AvatarForcing] + image → silent video
                                      │
                              [ffmpeg merge] + PCM → output.mp4
```

In [ ]:
!git clone https://github.com/MoshiHead/Moshi-AvatarForcing-bridge-v4.git

In [ ]:
%cd /workspace/Moshi-AvatarForcing-bridge-v4

---
## 📦 Cell 1 — System Dependencies

In [ ]:
%%bash
apt-get update -qq && apt-get install -y -qq \
    ffmpeg libsndfile1 sox git wget curl 2>/dev/null
echo '✅ ffmpeg + system libs installed'

## 📦 Cell 2 — Python Dependencies

In [ ]:
# %%bash
# # PyTorch (CUDA 12.1 — change cu121 → cu118 for older GPUs)
# pip uninstall -y torch torchvision torchaudio
# pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
# echo '✅ PyTorch installed'

!pip uninstall -y torch torchvision torchaudio flash-attn
# Install Torch 2.6 + CUDA 12.4
!pip install torch==2.6.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

In [ ]:
%%bash

pip install -q \
    "moshi==0.2.4" \
    "sphn>=0.1.4" \
    "sentencepiece>=0.1.99" \
    "transformers==4.41.2" \
    "accelerate==0.31.0" \
    "diffusers==0.30.0" \
    "peft==0.13.0" \
    "ftfy" \
    "easydict" \
    "safetensors>=0.4.0" \
    "huggingface_hub>=0.23.0" \
    "omegaconf>=2.3.0" \
    "einops>=0.7.0" \
    "soundfile>=0.12.1" \
    "pyyaml>=6.0" \
    "pillow>=10.0" \
    "decord>=0.6.0" \
    "websockets>=12.0" \
    "lmdb>=1.4.0" \
    "pandas>=2.0" \
    "opencv-python-headless>=4.8" \
    "pyworld>=0.3.4" \
    "tqdm>=4.66" \
    "ipywidgets>=8.0" \
    "numpy" \
    "scipy" \
    "imageio" \
    "imageio-ffmpeg" \
    "av" \
    "rotary-embedding-torch"

echo '✅ Python dependencies installed'

In [ ]:
# %%bash
# # Install Moshi package from local source
# cd ./moshi-inference && pip install -e . -q
# echo '✅ Moshi package installed'

In [ ]:
!pip install pytorch-lightning torchmetrics -q

In [ ]:
!pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl --no-build-isolation

In [ ]:
import flash_attn
print(flash_attn.__version__)

In [ ]:
%cd /workspace/Moshi-AvatarForcing-bridge-v4

## 📥 Cell 3 — Download Model Weights

In [ ]:
import os
from huggingface_hub import snapshot_download, hf_hub_download

os.makedirs('./checkpoints', exist_ok=True)
os.makedirs('./wan_models', exist_ok=True)

# ── Wan2.1-T2V-1.3B backbone ─────────────────────────────────────────────────
print('Downloading Wan2.1-T2V-1.3B (backbone for AvatarForcing) …')
snapshot_download(
    repo_id   = 'Wan-AI/Wan2.1-T2V-1.3B',
    local_dir = './AvatarForcing-inference/wan_models/Wan2.1-T2V-1.3B',
    ignore_patterns = ['*.md', '*.txt'],
)
print('✅ Wan2.1-T2V-1.3B ready')

In [ ]:
from huggingface_hub import snapshot_download

print("Downloading full AvatarForcing repo...")

snapshot_download(
    repo_id="lycui/AvatarForcing",
    local_dir="./checkpoints",
    local_dir_use_symlinks=False  # optional, avoids symlink issues
)

print("✅ Full repo downloaded")

In [ ]:
# ── Bridge checkpoint (your trained model) ────────────────────────────────────
# Option A: if you have uploaded it to HuggingFace:
hf_hub_download(repo_id='Darknsu/new-bridge-model-12-layer-v1',
                filename='bridge_best.pt',
                repo_type = 'dataset' ,
                local_dir='./checkpoints')

# Option B: upload manually via the RunPod file browser or scp

BRIDGE_CKPT = './checkpoints/bridge_best.pt'
import os
if os.path.exists(BRIDGE_CKPT):
    print(f'✅ Bridge checkpoint found: {BRIDGE_CKPT}')
else:
    print(f'⚠️  Bridge checkpoint NOT found at {BRIDGE_CKPT}')
    print('   Please upload bridge_best.pt before running inference.')

## ⚙️ Cell 4 — Configure Paths

In [ ]:
import sys, os

WORKSPACE = '/workspace/Moshi-AvatarForcing-bridge-v4'

# Repository roots
MOSHI_ROOT   = f'{WORKSPACE}/moshi-inference'
BRIDGE_ROOT  = f'{WORKSPACE}/moshi-wav2vec-bridge'
AF_ROOT      = f'{WORKSPACE}/AvatarForcing-inference'

# Add to Python path
for p in [WORKSPACE, MOSHI_ROOT, BRIDGE_ROOT, AF_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Checkpoint paths ──────────────────────────────────────────────────────────
BRIDGE_CKPT   = f'{WORKSPACE}/checkpoints/bridge_best.pt'
BRIDGE_CONFIG = f'{BRIDGE_ROOT}/config.yaml'
AF_CKPT       = f'{WORKSPACE}/checkpoints/model.pt'
AF_CONFIG     = f'{AF_ROOT}/configs/avatarforcing.yaml'

# ── Moshi model (auto-downloads from HuggingFace) ─────────────────────────────
MOSHI_HF_REPO = 'kyutai/moshiko-pytorch-q8'

# ── Generation settings — MUST match AvatarForcing avatarforcing.yaml ─────────
TEACHER_LEN       = 80   # even number; output frames at 25 Hz
                         # bridge needs TEACHER_LEN/2 = 40 Moshi steps ≈ 3.2 s
NUM_OUTPUT_FRAMES = 21   # video frames AvatarForcing generates
FPS               = 25   # output video FPS

DEVICE = 'cuda'

print('Configuration:')
print(f'  TEACHER_LEN      = {TEACHER_LEN}  ({TEACHER_LEN/2} Moshi steps = {TEACHER_LEN/25:.1f}s)')
print(f'  NUM_OUTPUT_FRAMES= {NUM_OUTPUT_FRAMES}')
print(f'  FPS              = {FPS}')
print(f'  DEVICE           = {DEVICE}')

## 🚀 Cell 5 — Load All Models

In [ ]:
import torch
from omegaconf import OmegaConf
from pathlib import Path

from unified_pipeline.pipeline import UnifiedMoshiAvatarPipeline

# ── Load AvatarForcing config (exact same way as inference.py) ────────────────
af_config = OmegaConf.load(AF_CONFIG)
default_cfg_path = Path(AF_CONFIG).parent / 'default_config.yaml'
if default_cfg_path.exists():
    default_cfg = OmegaConf.load(str(default_cfg_path))
    af_config   = OmegaConf.merge(default_cfg, af_config)

# Override teacher_len to match our settings
OmegaConf.update(af_config, 'data.teacher_len', TEACHER_LEN, merge=True)

print('AvatarForcing config loaded. Building pipeline …')

# ── Build + load all models ───────────────────────────────────────────────────
pipeline = UnifiedMoshiAvatarPipeline(
    bridge_checkpoint  = BRIDGE_CKPT,
    generator_ckpt     = AF_CKPT,
    af_config          = af_config,
    bridge_config      = BRIDGE_CONFIG,
    moshi_hf_repo      = MOSHI_HF_REPO,
    teacher_len        = TEACHER_LEN,
    num_output_frames  = NUM_OUTPUT_FRAMES,
    device             = DEVICE,
    dtype              = torch.bfloat16,
)

print('\n✅ All models loaded and ready!')

## 🎬 Cell 6 — Run Single Inference

In [ ]:
%cd /workspace/Moshi-AvatarForcing-bridge-v4

In [ ]:
# ── Set your input files ──────────────────────────────────────────────────────
USER_AUDIO = './example/audio.mp3'     # ← your speech input
IMAGE_PATH = './example/image.png'  # ← reference face image
OUTPUT_MP4 = './output.mp4'

# Optional custom prompt (set to None to use default)
PROMPT = ' A bald, feature-minimal synthetic male face with blue eyes is centered against a pure white void. The head is completely static and locked in a fixed frontal position; there is zero tilting, shifting, or rotation. The facial expression is strictly neutral and devoid of emotion. The eyes are fixed forward in a steady, unmoving gaze, with all ocular jitter eliminated except for occasional, simple blinks. Speech is rendered with mechanical precision, where sharp lip-syncing and jaw articulation are the only active movements on the face. All other facial features remain perfectly frozen. Lighting is flat and clinical, highlighting skin textures and freckles without creating shadows. The output must maintain absolute temporal stability, ensuring the facial structure remains rigid while the lips move with robotic accuracy to the audio.'

import os
assert os.path.exists(USER_AUDIO), f'Audio not found: {USER_AUDIO}'
assert os.path.exists(IMAGE_PATH), f'Image not found: {IMAGE_PATH}'
print(f'▶ Input audio : {USER_AUDIO}')
print(f'▶ Input image : {IMAGE_PATH}')
print(f'▶ Output      : {OUTPUT_MP4}')

In [ ]:
import torch

with torch.no_grad():
    output_path = pipeline.run(
        user_audio_path = USER_AUDIO,
        image_path      = IMAGE_PATH,
        output_path     = OUTPUT_MP4,
        prompt          = PROMPT,
        fps             = FPS,
    )

print(f'\n✅ Done! Output saved to: {output_path}')

## 🎥 Cell 7 — Preview Output

In [ ]:
from IPython.display import Video, display
import os

if os.path.exists(OUTPUT_MP4):
    size_mb = os.path.getsize(OUTPUT_MP4) / 1e6
    print(f'File size: {size_mb:.2f} MB')
    display(Video(OUTPUT_MP4, embed=True, width=640))
else:
    print(f'⚠️  Output not found at {OUTPUT_MP4}')

## 🔁 Cell 8 — Batch Processing

In [ ]:
import glob, os, torch

AUDIO_GLOB = './inputs/*.wav'
FACE_IMAGE = './reference_face.jpg'
OUTPUT_DIR = './outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

audio_files = sorted(glob.glob(AUDIO_GLOB))
print(f'Found {len(audio_files)} audio file(s)')

with torch.no_grad():
    for i, audio_path in enumerate(audio_files):
        stem = os.path.splitext(os.path.basename(audio_path))[0]
        out  = os.path.join(OUTPUT_DIR, f'{stem}.mp4')
        print(f'\n[{i+1}/{len(audio_files)}] {audio_path}')
        try:
            pipeline.run(
                user_audio_path = audio_path,
                image_path      = FACE_IMAGE,
                output_path     = out,
                fps             = FPS,
            )
            print(f'  ✅ → {out}')
        except Exception as e:
            print(f'  ❌ Failed: {e}')

print('\n✅ Batch done')

## 🔧 Cell 9 — Shape Sanity Check

In [ ]:
# Verify bridge output exactly matches AvatarForcing's expected audio_emb shape
import torch, sys
sys.path.insert(0, BRIDGE_ROOT)
from unified_pipeline.bridge_runner import BridgeRunner

br = BridgeRunner(BRIDGE_CKPT, BRIDGE_CONFIG, teacher_len=TEACHER_LEN, device=DEVICE)
br.reset()

result = None
for _ in range(br.mimi_frames_needed):
    result = br.push_tokens(torch.randint(0, 2048, (1, 8)))

expected = (TEACHER_LEN + 1, 14 * 768)  # (81, 10752)
print(f'Bridge output shape : {result.shape}')
print(f'Expected shape      : {expected}')
assert tuple(result.shape) == expected, f'SHAPE MISMATCH: {result.shape} != {expected}'
print('\n✅ Shape verified — bridge output matches AvatarForcing batch["audio_emb"]')

---
## 📐 Architecture Reference

| Stage | Input | Output | Rate |
|---|---|---|---|
| Moshi Mimi Encoder | 24kHz audio | 8 codebook tokens | 12.5 Hz |
| Moshi Helium LM | 8 user tokens | 8 response tokens + 1 text | 12.5 Hz |
| Moshi Mimi Decoder | 8 response tokens | raw PCM | 24kHz |
| **Bridge (new)** | 8 response tokens | (T+1, **10752**) audio_emb | 25 Hz |
| AvatarForcing | audio_emb + image | silent video (T, H, W, 3) | 25 FPS |
| ffmpeg merge | silent video + PCM | **output.mp4** | — |

**audio_emb = `last_hidden_state` + 13 × `hidden_states` = 14 × 768 = 10752 dim**  
This is identical to `facebook/wav2vec2-base-960h` with `output_hidden_states=True`.

**AvatarForcing code changes = ZERO.**  
Only the source of `batch['audio_emb']` is replaced.